# Gemma3: KerasHub export and Tunix SFT

Draft integration example for [keras-hub #3236](https://github.com/keras-team/keras-hub/issues/3236).

This notebook reuses KerasHub's Transformers export API and Tunix's Gemma3 safetensors loader. The CPU example uses a **small random model**, with local and global attention, to verify logits parity, parameter counts, SFT parameter updates, and save/reload of trained parameters. It is not a pretrained 270M model quality demonstration.

The saved output was executed locally on CPU with Python 3.12. Use a Python 3.12 Colab runtime to rerun it. The optional pretrained command at the end requires authorized Gemma checkpoint access and has not been validated here.

In [1]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

COMMIT = "c029da993cb97b876e4855e32ffd4d93195a1915"
assert sys.version_info >= (3, 12), "Use a Python 3.12 runtime."
local_repo = os.environ.get("TUNIX_REVIEW_REPO")
if local_repo:
    repo = Path(local_repo).resolve()
else:
    repo = Path(tempfile.mkdtemp()) / "keras-hub"
    subprocess.run(
        [
            "git",
            "clone",
            "-q",
            "https://github.com/bzantium/keras-hub.git",
            str(repo),
        ],
        check=True,
    )
    subprocess.run(["git", "checkout", "-q", COMMIT], cwd=repo, check=True)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-e",
            str(repo),
            "-r",
            str(repo / "tools/tunix/requirements.txt"),
        ],
        check=True,
    )
head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=repo, text=True
).strip()
assert head == COMMIT, head
print("KerasHub commit:", head)
print("Python:", sys.version.split()[0])
os.environ["KERAS_BACKEND"] = "jax"
os.environ["JAX_PLATFORMS"] = "cpu"
sys.path.insert(0, str(repo))

KerasHub commit: c029da993cb97b876e4855e32ffd4d93195a1915
Python: 3.12.13


## CPU integration check

The six-layer model includes both local and global attention. Its local window is shorter than the input sequence. The same unpadded token IDs are supplied to KerasHub and Tunix. Training uses two SGD steps on a synthetic prompt/completion batch, with loss only on completion tokens.

Assertions check parameter-count equality, float32 logits with `atol=1e-4, rtol=1e-4`, finite and decreasing loss, changed parameters, and exact logits after restoring the trained Orbax checkpoint.

In [2]:
import json

output_dir = Path(tempfile.mkdtemp(prefix="gemma3-tunix-"))
env = dict(
    os.environ, KERAS_BACKEND="jax", JAX_PLATFORMS="cpu", PYTHONPATH=str(repo)
)
result = subprocess.run(
    [
        sys.executable,
        str(repo / "tools/tunix/finetune_gemma3.py"),
        "--output_dir",
        str(output_dir),
    ],
    cwd=repo,
    env=env,
    capture_output=True,
    text=True,
)
if result.returncode:
    print(result.stdout)
    print(result.stderr)
    result.check_returncode()
print(result.stdout)
results = json.loads((output_dir / "results.json").read_text())
assert results["trained_checkpoint_logits_match"]
print("All integration checks passed.")

{
  "preset": "random Gemma3 CPU smoke check",
  "versions": {
    "keras": "3.15.1",
    "google-tunix": "0.1.7",
    "jax": "0.11.0",
    "flax": "0.12.8",
    "safetensors": "0.7.0"
  },
  "devices": [
    "cpu:0"
  ],
  "parameters": 58336,
  "max_absolute_logit_error": 1.5497207641601562e-06,
  "mean_absolute_logit_error": 3.280103442193649e-07,
  "training_steps": 2,
  "initial_loss": 4.600063323974609,
  "final_loss": 4.286632537841797,
  "max_logit_change_after_training": 0.5108265280723572,
  "max_parameter_change": 0.004188813269138336,
  "trained_checkpoint_logits_match": true
}

All integration checks passed.


## Reject empty target masks

An all-false mask and a mask active only at the first token have no next-token targets. Both must raise an error before evaluating the model. A single active next-token target must still produce the expected cross-entropy.

In [3]:
from unittest import mock

import jax.numpy as jnp
import numpy as np
from tunix.sft.peft_trainer import TrainingInput

from tools.tunix import finetune_gemma3 as example

tokens = np.array([[2, 3, 4, 5, 6, 7, 8, 9]], dtype="int32")
for mask in (
    np.zeros_like(tokens, dtype=bool),
    np.array([[True] + [False] * 7]),
):
    batch = TrainingInput(input_tokens=tokens, input_mask=mask)
    with mock.patch.object(example, "logits") as forward:
        try:
            example.loss(None, batch)
        except ValueError as error:
            assert "at least one target token" in str(error)
        else:
            raise AssertionError("Empty targets were accepted.")
        forward.assert_not_called()

batch = TrainingInput(
    input_tokens=tokens, input_mask=np.array([[False] * 7 + [True]])
)
with mock.patch.object(example, "logits", return_value=jnp.zeros((1, 8, 64))):
    np.testing.assert_allclose(example.loss(None, batch), np.log(64), rtol=1e-6)
print("Empty and first-token-only masks rejected; single-target loss verified.")

/tmp/keras-hub-tunix-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


I0000 00:00:1794141133.135967  391602 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794141133.136818  391602 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1794141134.069078  391602 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794141134.069505  391602 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Empty and first-token-only masks rejected; single-target loss verified.


## Optional: real Gemma3 270M IT checkpoint

The real checkpoint was inaccessible to the account used for the saved run. This command is provided for validation after accepting the Gemma model terms and authenticating with a Hugging Face account that has access. Authentication tokens should not be written into this notebook.

From the cloned repository, run:

```shell
KERAS_BACKEND=jax PYTHONPATH=. python tools/tunix/finetune_gemma3.py \
    --preset hf://google/gemma-3-270m-it \
    --output_dir /tmp/gemma3-270m-tunix
```

This mode tokenizes a short instruction and completion, performs the same parity checks and two full-fine-tuning steps, then reloads the trained parameters. It uses float32 and holds multiple model copies for verification, so allow more memory than the checkpoint size alone. For an accelerator run, install the matching JAX accelerator packages first.

Gemma4, LoRA, distributed training, padded batches, and importing Tunix-trained weights into KerasHub are outside this initial example.